# LC 143 — Reorder List
**Difficulty:** Medium | **Pattern:** Linked List / Find-Mid + Reverse + Merge

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Split the list in half with slow/fast
pointers, reverse the second half in-place, then weave the two halves
together by alternating nodes. Three clean sub-problems — each one
a classic pattern on its own.
</div>

## Official Problem Statement
You are given the head of a singly linked-list:
`L0 → L1 → … → Ln-1 → Ln`

Reorder it to: `L0 → Ln → L1 → Ln-1 → L2 → Ln-2 → …`

You may not modify the values in the list's nodes.
Only node rearrangement is allowed. Do it **in-place**.

**Example 1:**
```
Input:  [1, 2, 3, 4]
Output: [1, 4, 2, 3]
```
**Example 2:**
```
Input:  [1, 2, 3, 4, 5]
Output: [1, 5, 2, 4, 3]
```
**Constraints:**
- Number of nodes: `[1, 50000]`
- `1 <= Node.val <= 1000`

## What This Is Actually Asking
Imagine folding the list in half like a book:

```
  1  2  3  4  5
  ^           ^   <- fold
  1  5  2  4  3
```

The first half is read left-to-right; the second half is read
right-to-left. You interleave one node from each half in turn.

You cannot use an array or extra list — all pointer rewiring
must happen inside the existing nodes.

## Walk Through an Example by Hand
List: `1 → 2 → 3 → 4 → 5`

**Step 1 — Find mid (slow/fast):**
```
slow, fast = head, head
Move: slow=2,fast=3 → slow=3,fast=5
fast.next is None → stop. mid = slow = 3
```

**Step 2 — Reverse second half (from mid.next):**
```
First half:  1 → 2 → 3 → None  (cut at mid)
Second half: 4 → 5 → None
After reverse: 5 → 4 → None
```

**Step 3 — Merge alternating:**
```
p1=1, p2=5
  1.next=5, 5.next=2  → 1→5→2
p1=2, p2=4
  2.next=4, 4.next=3  → 1→5→2→4→3
p2=None → done
```
Result: `1 → 5 → 2 → 4 → 3`

## The Picture
```
ORIGINAL:

  [1] --> [2] --> [3] --> [4] --> [5] --> None


STEP 1 — Find mid with slow/fast pointers:

  s           s           s
  [1] --> [2] --> [3] --> [4] --> [5] --> None
  f                   f               f
  (fast moves 2 at a time; slow stops at mid)

  mid = [3]   Cut: mid.next = None

  First half:   [1] --> [2] --> [3] --> None
  Second half:  [4] --> [5] --> None


STEP 2 — Reverse second half:

  Before: [4] --> [5] --> None
  After:  [5] --> [4] --> None


STEP 3 — Merge alternating (p1 from left, p2 from right):

  p1=[1]  p2=[5]
  p1=[2]  p2=[4]
  p1=[3]  p2=None  <- stop

  [1] --> [5] --> [2] --> [4] --> [3] --> None
```

## When To Use This Pattern
Use the **Find-Mid + Reverse + Merge** pattern when:

- You must interleave the front and back halves of a list.
- You need to compare or combine nodes from opposite ends.
- You want **O(1) space** with no auxiliary data structures.

This pattern appears across several problems:
- Palindrome Linked List (LC 234) — compare halves
- Reorder List (LC 143) — interleave halves
- Sort List (LC 148) — merge-sort uses find-mid + merge

Recognizing the three sub-problems as **independent building
blocks** is the key insight.

## The Approach
**Algorithm: Find-Mid → Reverse Second Half → Merge**

**Step 1 — Find the middle node:**
- Use slow (moves 1) and fast (moves 2) pointers.
- When `fast` or `fast.next` is `None`, `slow` is at mid.
- Cut the list: `mid.next = None`.

**Step 2 — Reverse the second half:**
- Apply the classic three-pointer reversal on `second_head`.
- Returns the new head of the reversed second half.

**Step 3 — Merge alternating:**
- `p1` walks the first half, `p2` walks the reversed second.
- At each step: save `p1.next` and `p2.next`, then
  wire `p1.next = p2`, `p2.next = next_p1`.
- Advance both pointers.
- Stop when `p2` is exhausted.

**Complexity:** O(n) time, O(1) space.

In [ ]:
from typing import Optional


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val; self.next = next


def make_list(vals):
    dummy = ListNode(0); cur = dummy
    for v in vals:
        cur.next = ListNode(v); cur = cur.next
    return dummy.next


def to_list(head):
    r = []
    while head:
        r.append(head.val); head = head.next
    return r

In [ ]:
def test_harness(func):
    cases = [
        ([1, 2, 3, 4],    [1, 4, 2, 3]),
        ([1, 2, 3, 4, 5], [1, 5, 2, 4, 3]),
        ([1],             [1]),
        ([1, 2],          [1, 2]),
        ([1, 2, 3],       [1, 3, 2]),
    ]
    passed = 0
    for i, (inp, exp) in enumerate(cases):
        head = make_list(inp)
        func(head)  # modifies in-place, no return value
        result = to_list(head)
        status = "PASSED" if result == exp else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"  Case {i+1} {status}: "
                f"inp={inp} got={result} exp={exp}"
            )
    total = len(cases)
    print(
        f"\nResults: {passed}/{total} passed "
        + ("ALL GOOD!" if passed == total else "FIX NEEDED")
    )

In [ ]:
def reorder_list(head: Optional[ListNode]) -> None:
    """
    Reorder list in-place: L0->Ln->L1->Ln-1->L2->...

    Strategy: Three-step decomposition.
      1. Find mid with slow/fast pointers; cut list in half.
      2. Reverse second half using three-pointer technique.
      3. Merge two halves by alternating nodes.

    Modifies the list in-place; does not return a value.

    Args:
        head: Head of the singly linked list.

    Time:  O(n)  — three linear passes
    Space: O(1)  — constant extra pointers
    """
    # TODO: implement
    print(f"[DEBUG] head val = "
          f"{head.val if head else None}")
    pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(reorder_list)

## Complexity
| Dimension | Value | Reason |
|-----------|-------|--------|
| Time      | O(n)  | Three passes: find-mid, reverse, merge |
| Space     | O(1)  | Only a handful of pointer variables |

**Common wrong approach:** Collecting all values into an array
and rebuilding. This is O(n) space — violates the in-place
constraint and shows you missed the pointer pattern.

## Real World Connection
**Interleaved data streams:**
Audio/video players sometimes interleave frames from the
beginning and end of a buffer to create crossfade effects.
Reordering a linked list is the algorithmic core of that
operation when the buffer is stored as a linked sequence.

**Playlist reshuffling:**
Streaming apps rearrange playlists so that popular songs
alternate with less-played ones. If the playlist is a
linked list sorted by play count, this exact reorder
pattern (first + last + second + second-last…) distributes
content evenly across the listening session.

**Memory compaction:**
OS memory managers sometimes interleave free blocks from
the front and back of a free-list to maximize contiguous
space — conceptually the same reorder operation.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra